# dlt Workshop Homework — Solution

Uses Pydantic AI agent (Groq llama-3.3-70b-versatile) instrumented with Logfire, traces pulled into DuckDB via dlt.

## Setup

Run `main_solution.py` first to send traces to Logfire, then run `logfire_pipeline.py` to pull into DuckDB.

In [ ]:
import subprocess
# Step 1: Run agent to send traces to Logfire
result = subprocess.run(["uv", "run", "python", "main_solution.py"], capture_output=True, text=True)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

## Q1: How many spans does the trace produce?

Check the Logfire dashboard after running main_solution.py. The agent trace produces **4 spans**:
- `invoke_agent faq_agent` (root span)
- `chat llama-3.3-70b-versatile` (first LLM call — decides to search)
- `execute_tool search` (tool call)
- `chat llama-3.3-70b-versatile` (second LLM call — generates answer)

## Q2 & Q3: Pull traces into DuckDB with dlt

In [ ]:
import os
import dlt
import duckdb
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv

load_dotenv("/Users/dt00035/projects/personal/reference/llm-zoomcamp/.env")
load_dotenv(override=False)

READ_TOKEN = os.environ.get("LOGFIRE_READ_TOKEN", "")


@dlt.resource(name="spans", write_disposition="replace")
def logfire_spans():
    from logfire.query_client import LogfireQueryClient
    client = LogfireQueryClient(read_token=READ_TOKEN)
    min_ts = datetime.now(timezone.utc) - timedelta(days=7)
    result = client.query_json_rows(
        """
        SELECT trace_id, span_id, parent_span_id, span_name,
               start_timestamp, end_timestamp, duration, attributes
        FROM records
        ORDER BY start_timestamp DESC
        LIMIT 1000
        """,
        min_timestamp=min_ts,
    )
    yield from result["rows"]


@dlt.source(name="logfire_traces")
def logfire_source():
    yield logfire_spans()


pipeline = dlt.pipeline(
    pipeline_name="logfire_pipeline",
    destination="duckdb",
    dataset_name="agent_traces",
)

info = pipeline.run(logfire_source())
print(info)

## Q2: How many tables are in the agent_traces schema?

In [ ]:
conn = duckdb.connect("logfire_pipeline.duckdb")

count = conn.execute("""
    SELECT COUNT(*) FROM information_schema.tables
    WHERE table_schema = 'agent_traces'
""").fetchone()[0]

print(f"Q2: Tables in agent_traces schema = {count}")

tables = conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'agent_traces'
    ORDER BY table_name
""").fetchall()
print("Table names:", [t[0] for t in tables])

## Q3: What is the total input token usage?

In [ ]:
import pandas as pd

# Show all spans with token usage
df = pd.read_sql("""
    SELECT span_name, duration,
           attributes__gen_ai_usage_input_tokens as input_tokens,
           attributes__gen_ai_usage_output_tokens as output_tokens
    FROM agent_traces.spans
    ORDER BY start_timestamp
""", conn)

print(df.to_string(index=False))

total_input = df["input_tokens"].sum()
print(f"\nQ3: Total input tokens = {int(total_input)}")
print(f"    (per LLM call: {df[df['input_tokens'].notna()]['input_tokens'].tolist()})")